<img src="https://hilpisch.com/tpq_logo_bic.png" width="20%" align="right">

# Python & AI for Rates, Bonds, and Credit

## Chapter 6 · Curve Bootstrapping and Multi-Curve Logic

[Open in Google Colab](https://colab.research.google.com/github/yhilpisch/ratescode/blob/main/notebooks/ch06_curve_bootstrapping_multi_curve_logic.ipynb)

&copy; Dr. Yves J. Hilpisch<br>
AI-Powered by different LLMs<br>
The Python Quants GmbH | https://tpq.io<br>
https://hilpisch.com | https://linktr.ee/dyjh

## Learning goals

Curve construction turns market quotes into discount factors that must reprice the very instruments used to build them. In this lab you bootstrap a single-curve discount curve from annual par swap quotes, convert each solved node into a continuously compounded zero rate, and accept the curve only after the repricing residuals vanish. You then connect the single-curve par swap identity to the multi-curve framework, distinguishing the curve that discounts cash flows from the curve that projects floating coupons, and you state the payment frequency, accrual, and compounding conventions each number depends on.

## Prepare the companion repository

This setup cell works with a local checkout or a fresh Google Colab runtime. In Colab it clones the public companion repository, then sets `ROOT` so the chapter code and frozen data snapshots are found without manual path edits.

In [ ]:
from pathlib import Path
import importlib.util
import subprocess
import sys

REPO_URL = "https://github.com/yhilpisch/ratescode.git"
candidates = [Path.cwd(), Path.cwd().parent,
              Path("/content/07_ratescode")]
ROOT = next((p.resolve() for p in candidates
             if (p / "code").is_dir()
             and (p / "data").is_dir()), None)
if ROOT is None:
    ROOT = Path("/content/07_ratescode")
    subprocess.run(["git", "clone", "--depth", "1",
                    REPO_URL, str(ROOT)], check=True)

packages = {"numpy": "numpy", "pandas": "pandas",
            "matplotlib": "matplotlib"}
missing = [package for module, package in packages.items()
           if importlib.util.find_spec(module) is None]
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install",
                    *missing], check=True)
print(f"Repository: {ROOT}")
print(f"Data files: {len(list((ROOT / 'data').glob('*.csv')))}")


## Run the chapter example

The script bootstraps a single-curve discount curve from four annual par swap quotes (maturities one to four years, par rates 3.60%, 3.80%, 3.95%, and 4.05%), derives the continuously compounded zero rate at each node, and reprices every input swap. The printed curve table lists maturity, swap rate, discount factor, and zero rate; the residual table then shows fixed-leg minus floating-leg pricing errors, which vanish to floating-point precision when the bootstrap is correct.

In [ ]:
SCRIPT = ROOT / "code" / "ch06_curve_bootstrapping.py"
print(SCRIPT.relative_to(ROOT))

result = subprocess.run(
    [sys.executable, str(SCRIPT)],
    cwd=ROOT,
    capture_output=True,
    text=True,
    check=True,
)
print(result.stdout)

## Interpretation checklist

1. Confirm the sequential logic: the 1-year quote alone solves the first discount factor, and each later quote solves only its own final node from the discounts already found.
2. Verify one node by hand, for example D(0,2) = (1 - 0.0380 * 0.965251) / (1 + 0.0380) = 0.928054 under annual payments with accrual fractions of one.
3. Check that every zero rate sits close to, and generally below, its par swap rate, and explain why continuous zero rates reproduce annually compounded par quotes.
4. Confirm the repricing residuals print as zero (or -0.0) at ten-decimal rounding; that is the acceptance test for the calibration, not evidence about market data quality.
5. Note the simplification: a single curve both discounts and projects, while a production setup would separate OIS discounting from tenor-specific projection curves.

In [ ]:
import pandas as pd

DATA = ROOT / "data"
chapter_files = sorted(DATA.glob("ch06_*.csv"))
summary = pd.DataFrame([
    {"file": path.name,
     "rows": len(frame := pd.read_csv(path)),
     "columns": len(frame.columns),
     "missing_cells": int(frame.isna().sum().sum()),
     "fields": str(list(frame.columns))}
    for path in chapter_files
])
summary


## Mini-lab: Bootstrap two zero rates from par swaps

Bootstrapping solves one unknown at a time: each par swap quote adds one curve node. This lab starts from two illustrative annual par swap quotes, solves the 1-year and 2-year discount factors sequentially, and reports the continuously compounded zero rates they imply. The final line reprices both swaps to confirm the solved nodes give back the input rates.

In [ ]:
import numpy as np

par_rates = np.array([0.036, 0.038])  # annual par swap rates (decimals)
maturities = np.array([1.0, 2.0])  # fixed-leg payment times (years)

discounts = []  # bootstrapped discount factors
for rate in par_rates:
    known_pv = sum(discounts)  # present value of earlier coupons
    discounts.append((1 - rate * known_pv) / (1 + rate))
discounts = np.array(discounts)

zero_rates = -np.log(discounts) / maturities  # continuous zero rates
repriced = (1 - discounts) / np.cumsum(discounts)  # par repricing check
print("discounts:", discounts.round(6))
print("zero rates:", zero_rates.round(6))
print("repriced par rates:", repriced.round(6))

### What to observe

The solved nodes are 0.965251 and 0.928054, implying zero rates of 3.5367% and 3.7332% with continuous compounding. The repriced par rates come back as exactly 3.6000% and 3.8000%, so the curve reproduces its own inputs. Each quote added exactly one node; nothing was fit twice.

## Exercises

1. **Extend the bootstrap.** Add the 3-year par rate of 3.95% to the mini-lab inputs and solve the third discount factor. *Hint:* the known-coupon term is the new rate times the sum of the two solved discounts.
2. **Verify a node by hand.** Reproduce the 4-year discount factor of 0.852738 from the script's curve table with a calculator. *Hint:* the numerator subtracts 0.0405 times the sum of the first three discounts.
3. **Reprice all four quotes.** Rebuild the script's residual table and confirm every entry rounds to zero at ten decimals. *Hint:* in the single-curve setting the floating leg equals one minus the final discount factor.
4. **Stress one quote.** Move the 2-year par rate by five basis points and report which discount factors and zero rates change. *Hint:* only nodes at and beyond the changed quote should move.
5. **Separate the curves.** Explain in two sentences what the discount curve and the projection curve each answer in a multi-curve valuation. *Hint:* present values versus projected floating coupons.

For more detail, see Chapter 6 of *Python & AI for Rates, Bonds, and Credit*.

<img src="https://hilpisch.com/tpq_logo_bic.png" width="20%" align="right">